In [1]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
import json

In [ ]:
df = pd.read_csv('../../../data/processed/land_dataset_final_v3.csv')

In [3]:
with open('commune_nested_with_min_max.json', 'r') as file:
    data = json.load(file)

In [4]:
def get_min_max(row):
    city = row['address_subdivision']
    district = row['address_locality']
    commune = row['address_line_2']
    try:
        commune_list = data[city][district]
        for c in commune_list:
            if c['commune'] == commune:
                return pd.Series({'commune_min': c['min'], 'commune_max': c['max']})
    except KeyError:
        pass
    return pd.Series({'commune_min': None, 'commune_max': None})

In [5]:
df[['commune_min', 'commune_max']] = df.apply(get_min_max, axis=1)

In [6]:
# Define columns
distance_columns = ['near_Koh_Pich_in_km', 'near_Russian_Market_in_km', 'near_AEON_Mall_1_in_km', 'near_AEON_Mall_2_in_km', 'near_AEON_Mall_3_in_km', 'near_Bassac_Lane_in_km', 'near_Koh_Norea_in_km', 'near_Camko_City_in_km', 'near_Olympic_Stadium_in_km', 'near_Phsar_Tmey_in_km', 'near_Boeng_Keng_Kang_1_in_km', 'near_Wat_Phnom_in_km', 'near_Chroy_Changvar_Bridge_in_km', 'near_Vattanac_Tower_in_km', 'near_Royal_Palace_in_km', 'near_Sisowath_Riverside_Park_in_km', 'near_Phnom_Penh_Airport_in_km', 'near_Phsar_Chas_in_km', 'near_Phsar_kandal_in_km']
amenity_columns = ['n_cafe_5km', 'n_gas_station_5km', 'n_hospital_5km', 'n_hotel_5km', 'n_mart_5km', 'n_pre_school_5km', 'n_secondary_school_5km', 'n_primary_school_5km', 'n_university_5km', 'n_seven_eleven_5km', 'n_resturant_5km', 'n_super_market_5km', 'n_borey_5km', 'n_bank_5km', 'n_atm_5km']
road_columns = ['f_bridleway', 'f_corridor', 'f_cycleway', 'f_disused', 'f_footway', 'f_motorway', 'f_path', 'f_pedestrian', 'f_primary', 'f_residential', 'f_road', 'f_secondary', 'f_service', 'f_steps', 'f_tertiary', 'f_track', 'f_trunk', 'f_trunk_link', 'f_unclassified', 'f_unused']
ROAD_TYPE_WEIGHTS = {
    'f_residential': 1.0, 'f_pedestrian': 0.9, 'f_cycleway': 0.85, 'f_footway': 0.8,
    'f_primary': 0.7, 'f_secondary': 0.65, 'f_tertiary': 0.6, 'f_service': 0.55,
    'f_trunk': 0.4, 'f_trunk_link': 0.4, 'f_motorway': 0.3, 'f_unclassified': 0.5,
    'f_track': 0.2, 'f_path': 0.25, 'f_steps': 0.3, 'f_disused': 0.1,
    'f_unused': 0.1, 'f_corridor': 0.15, 'f_bridleway': 0.2, 'f_road': 0.5
}

In [7]:
DISTANCE_WEIGHT = 0.40
ROAD_WEIGHT = 0.35
AMENITY_WEIGHT = 0.25

AMENITY_WEIGHTS = {
    'n_hospital_5km': 1.7, 'n_bank_5km': 1.6, 'n_super_market_5km': 1.5, 
    'n_borey_5km': 1.5, 'n_university_5km': 1.4, 'n_secondary_school_5km': 1.3,
    'n_primary_school_5km': 1.3, 'n_pre_school_5km': 1.2, 'n_gas_station_5km': 1.2,
    'n_resturant_5km': 1.1, 'n_hotel_5km': 1.1, 'n_atm_5km': 1.0,
    'n_seven_eleven_5km': 1.0, 'n_mart_5km': 0.9, 'n_cafe_5km': 0.8
}

DISTANCE_DECAY = {
    'near_Vattanac_Tower_in_km': 0.7, 'near_Sisowath_Riverside_Park_in_km': 0.7,
    'near_Royal_Palace_in_km': 0.7, 'near_AEON_Mall_1_in_km': 0.6,
    'near_AEON_Mall_2_in_km': 0.6, 'near_Koh_Pich_in_km': 0.65,
    'near_Boeng_Keng_Kang_1_in_km': 0.6, 'near_Wat_Phnom_in_km': 0.55,
    'near_Chroy_Changvar_Bridge_in_km': 0.5, 'near_Phnom_Penh_Airport_in_km': 0.4,
    'near_Russian_Market_in_km': 0.45, 'near_AEON_Mall_3_in_km': 0.45,
    'near_Koh_Norea_in_km': 0.5, 'near_Camko_City_in_km': 0.45,
    'near_Olympic_Stadium_in_km': 0.4, 'near_Bassac_Lane_in_km': 0.4,
    'near_Phsar_Tmey_in_km': 0.35, 'near_Phsar_Chas_in_km': 0.35,
    'near_Phsar_kandal_in_km': 0.3
}

In [8]:
def calculate_desirability_score(row):
    """Calculate 0-1 desirability score with PP-specific weighting"""
    # 1. Distance component (location premium)
    distance_scores = []
    for col in distance_columns:
        decay = DISTANCE_DECAY.get(col, 0.5)
        distance_scores.append(np.exp(-decay * row[col]))
    distance_score = np.mean(distance_scores)
    
    # 2. Amenity component (livability)
    amenity_scores = []
    for col in amenity_columns:
        weight = AMENITY_WEIGHTS.get(col, 1.0)
        amenity_scores.append(weight * np.log1p(row[col]))
    amenity_score = np.mean(amenity_scores) if amenity_scores else 0
    
    # 3. Road quality component (accessibility)
    road_scores = []
    for col in road_columns:
        if row[col] == 1:
            road_scores.append(ROAD_TYPE_WEIGHTS[col])
    road_score = np.mean(road_scores) if road_scores else 0.5
    
    # Weighted composite score
    composite_score = (
        DISTANCE_WEIGHT * distance_score +
        AMENITY_WEIGHT * amenity_score +
        ROAD_WEIGHT * road_score
    )
    
    # Add controlled noise (σ=3% of score range)
    noise = np.random.normal(loc=0, scale=0.03)
    noisy_score = composite_score + noise
    
    # Constrain to [0,1] range
    return max(0.01, min(0.99, noisy_score))

In [9]:
df['desirability_score'] = df.apply(calculate_desirability_score, axis=1)

In [10]:
def normalize_group(scores):
    min_score = scores.min()
    max_score = scores.max()
    range_size = max_score - min_score
    
    if range_size > 0:
        return (scores - min_score) / range_size
    else:
        return np.random.uniform(0.3, 0.7, len(scores))

df['desirability_score'] = df.groupby('address_line_2')['desirability_score'].transform(normalize_group)

In [11]:
def generate_price_per_m2(min_val, max_val, generated_prices, desirability_score, tolerance=0.01):
    """
    Generate price based on 3-tier location desirability with uniqueness constraint
    """
    # Handle case where min = max
    if min_val == max_val:
        center = min_val
        if center > 1000:
            variation = center * 0.15
        elif center > 100:
            variation = center * 0.20
        else:
            variation = center * 0.30
        min_val = max(0, center - variation)
        max_val = center + variation

    # Handle other edge cases
    if max_val <= min_val:
        return round(min_val, 2)

    total_range = max_val - min_val
    mean = (min_val + max_val) / 2

    # Define 3 price bands: low, medium, high
    if desirability_score >= 0.66:
        # High desirability
        band = (mean + 0.25 * total_range, max_val - 0.01 * total_range)
    elif desirability_score >= 0.33:
        # Medium desirability
        band = (mean - 0.1 * total_range, mean + 0.1 * total_range)
    else:
        # Low desirability
        band = (min_val + 0.01 * total_range, mean - 0.25 * total_range)

    # Ensure band is valid
    low_bound = max(min_val, band[0])
    high_bound = min(max_val, band[1])
    
    # Generate candidate price
    for _ in range(100):
        candidate = random.uniform(low_bound, high_bound)
        rounded = round(candidate, 2)

        # Check uniqueness
        if all(abs(rounded - existing) / total_range >= tolerance for existing in generated_prices):
            return rounded

    # Fallback to simple random in band
    return round(random.uniform(low_bound, high_bound), 2)

In [12]:
generated_prices = set()
def apply_generate_price(row):
    price = generate_price_per_m2(
        row['commune_min'],
        row['commune_max'],
        generated_prices,
        row['desirability_score']
    )
    generated_prices.add(price)
    return price

df['price_per_m2'] = df.apply(apply_generate_price, axis=1)

In [13]:
df.drop(columns=['desirability_score', 'commune_min', 'commune_max'], inplace=True)
df.dropna(inplace=True)
df['price'] = df['price_per_m2']*df['land_area']

In [14]:
numerical_features = [
    'land_area', 'n_floors', 'n_bedrooms', 'n_bathrooms',
    'near_Koh_Pich_in_km', 'near_Russian_Market_in_km', 'near_AEON_Mall_1_in_km', 
    'near_AEON_Mall_2_in_km', 'near_AEON_Mall_3_in_km', 'near_Bassac_Lane_in_km', 
    'near_Koh_Norea_in_km', 'near_Camko_City_in_km', 'near_Olympic_Stadium_in_km', 
    'near_Phsar_Tmey_in_km', 'near_Boeng_Keng_Kang_1_in_km', 'near_Wat_Phnom_in_km', 
    'near_Chroy_Changvar_Bridge_in_km', 'near_Vattanac_Tower_in_km', 
    'near_Royal_Palace_in_km', 'near_Sisowath_Riverside_Park_in_km', 
    'near_Phnom_Penh_Airport_in_km', 'near_Phsar_Chas_in_km', 'near_Phsar_kandal_in_km',
    'n_cafe_5km', 'n_gas_station_5km', 'n_hospital_5km', 'n_hotel_5km', 
    'n_mart_5km', 'n_pre_school_5km', 'n_secondary_school_5km', 
    'n_primary_school_5km', 'n_university_5km', 'n_seven_eleven_5km', 
    'n_resturant_5km', 'n_super_market_5km', 'n_borey_5km', 'n_bank_5km', 'n_atm_5km'
]


In [15]:
for feature in numerical_features:
    if feature in df.columns:
        # Calculate noise scale (1% of standard deviation)
        noise_scale = 0.02 * df[feature].std()
        
        # Skip if no variation (std=0)
        if noise_scale > 0:
            noise = np.random.normal(0, noise_scale, len(df))
            df[feature] += noise
            
            # Apply feature-specific constraints
            if 'near_' in feature:  # Distance features
                df[feature] = df[feature].clip(lower=0)  # Ensure non-negative
            elif feature == 'land_area':
                df[feature] = df[feature].clip(lower=10)  # Minimum land area
            elif feature.startswith('n_'):  # Count features
                df[feature] = df[feature].round().clip(lower=0)  # Round to integers


In [16]:
noise_factor = np.random.normal(1.0, 0.02, len(df))
df['price_per_m2'] *= noise_factor

In [17]:
df['price_per_m2'] *= np.random.uniform(0.7, 1.2, size=len(df))

In [18]:
df['price'] = df['land_area'] * df['price_per_m2']

In [19]:
df.to_csv('../../../data/processed/land_dataset_final_100k_noisy.csv', index=False)

In [20]:
min_price_per_m2 = df['price_per_m2'].min()
max_price_per_m2 = df['price_per_m2'].max()

# Print the results
print(f"Minimum price per m2: {min_price_per_m2}")
print(f"Maximum price per m2: {max_price_per_m2}")

Minimum price per m2: 14.426616455679692
Maximum price per m2: 12089.970208061868


In [21]:
df

,address_subdivision,address_locality,address_line_2,h_id,price_per_m2,land_area,price,longitude,latitude,near_Koh_Pich_in_km,...,f_road,f_secondary,f_service,f_steps,f_tertiary,f_track,f_trunk,f_trunk_link,f_unclassified,f_unused
0,Phnom Penh,Praek Pnov,Samraong,886580d009fffff,40.197109,130.076426,5228.696317,104.830384,11.717244,22.019637,...,0,0,0,0,0,0,0,0,0,0
1,Phnom Penh,Pur SenChey,Kamboul,8865846ea7fffff,61.775256,173.913010,10743.520701,104.741205,11.547118,20.875191,...,0,0,0,0,1,1,0,0,1,0
2,Phnom Penh,Chraoy Chongvar,Preaek Ta Sek,886580d26dfffff,292.569134,148.369890,43408.450185,104.881978,11.676204,14.951726,...,0,0,1,0,0,0,0,0,0,0
3,Phnom Penh,Pur SenChey,Ovlaok,8865846c3bfffff,103.369761,150.223707,15528.588632,104.759005,11.579001,20.157050,...,0,0,0,0,0,0,0,0,0,0
4,Phnom Penh,Pur SenChey,Boeng Thum,8865846521fffff,148.265688,95.702529,14189.401260,104.770540,11.482552,19.913356,...,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,Phnom Penh,Pur SenChey,Boeng Thum,886584652bfffff,109.736470,191.459694,21010.110971,104.769793,11.471359,20.055508,...,0,0,0,0,0,0,0,0,0,0
99996,Phnom Penh,Mean Chey,Chak Angrae Kraom,8865846109fffff,740.597726,107.848864,79872.623580,104.931755,11.486649,7.048524,...,0,0,1,0,0,0,0,0,0,0
99997,Phnom Penh,Dangkao,Prey Sa,8865846189fffff,817.868079,48.071351,39316.023140,104.869213,11.492956,9.845953,...,0,0,0,0,0,0,0,0,0,0
99998,Phnom Penh,Saensokh,Krang Thnong,8865846d49fffff,1283.382081,62.268717,79914.555763,104.831913,11.592815,12.092547,...,0,0,0,0,0,0,0,0,0,0


In [22]:
print("Dataset with noise saved successfully!")
print(f"Final dataset shape: {df.shape}")

Dataset with noise saved successfully!
Final dataset shape: (100000, 234)
